# Kidney Ultrasound/CT Classifier — Normal vs Abnormal

Run this notebook in **Google Colab** with a GPU runtime (Runtime → Change runtime type → GPU).

**Before running:** zip your organized dataset (from `organize_dataset.py`) and upload it to your Google Drive, e.g. as `MyDrive/kidney_dataset.zip`, with this structure inside:
```
kidney_dataset/
    Normal/      *.jpg
    Abnormal/    *.jpg
```

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import zipfile, os

ZIP_PATH = '/content/drive/MyDrive/kidney_dataset.zip'  # change if needed
EXTRACT_PATH = '/content/data'

os.makedirs(EXTRACT_PATH, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall(EXTRACT_PATH)

print(os.listdir(EXTRACT_PATH))

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

print('TensorFlow version:', tf.__version__)
print('GPU available:', tf.config.list_physical_devices('GPU'))

# IMPORTANT: point this to the folder that directly contains Normal/ and Abnormal/
DATA_DIR = '/content/data/kidney_dataset'  # adjust based on your zip's internal folder name
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

## 2. Load data and split (train/val/test = 70/15/15)

In [ ]:
full_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    labels='inferred',
    label_mode='binary',
    class_names=['Normal', 'Abnormal'],
    image_size=IMG_SIZE,
    batch_size=None,
    shuffle=True,
    seed=SEED
)

n = tf.data.experimental.cardinality(full_ds).numpy()
train_n = int(0.7 * n)
val_n = int(0.15 * n)

train_ds = full_ds.take(train_n)
remaining = full_ds.skip(train_n)
val_ds = remaining.take(val_n)
test_ds = remaining.skip(val_n)

print(f'Train: {train_n}, Val: {val_n}, Test: {n - train_n - val_n}')

train_ds = train_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

## 3. Data augmentation + preprocessing

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.05),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomContrast(0.1),
])

## 4. Handle class imbalance
Your Abnormal class has more images than Normal — compute class weights so the model doesn't get biased toward Abnormal.

In [ ]:
import pathlib
data_path = pathlib.Path(DATA_DIR)
n_normal = len(list((data_path / 'Normal').glob('*.jpg')))
n_abnormal = len(list((data_path / 'Abnormal').glob('*.jpg')))
total = n_normal + n_abnormal

class_weight = {
    0: total / (2 * n_normal),     # Normal
    1: total / (2 * n_abnormal),   # Abnormal
}
print('Normal:', n_normal, 'Abnormal:', n_abnormal)
print('Class weights:', class_weight)

## 5. Build the model (transfer learning — MobileNetV2)

In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False  # freeze for initial training

inputs = tf.keras.Input(shape=(224, 224, 3))
x = data_augmentation(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)
model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Precision(name='precision'), tf.keras.metrics.Recall(name='recall')]
)
model.summary()

## 6. Train (phase 1: frozen base)

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    class_weight=class_weight,
    callbacks=callbacks
)

## 7. Fine-tune (phase 2: unfreeze top layers of base model)

In [ ]:
base_model.trainable = True
# Freeze all but the last ~30 layers
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Precision(name='precision'), tf.keras.metrics.Recall(name='recall')]
)

history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    class_weight=class_weight,
    callbacks=callbacks
)

## 8. Plot training curves

In [ ]:
acc = history.history['accuracy'] + history_fine.history['accuracy']
val_acc = history.history['val_accuracy'] + history_fine.history['val_accuracy']
loss = history.history['loss'] + history_fine.history['loss']
val_loss = history.history['val_loss'] + history_fine.history['val_loss']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(acc, label='train'); axes[0].plot(val_acc, label='val')
axes[0].set_title('Accuracy'); axes[0].legend()
axes[1].plot(loss, label='train'); axes[1].plot(val_loss, label='val')
axes[1].set_title('Loss'); axes[1].legend()
plt.show()

## 9. Evaluate on the held-out test set

In [ ]:
y_true = []
y_pred_probs = []
for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_pred_probs.extend(preds.flatten())
    y_true.extend(labels.numpy().flatten())

y_true = np.array(y_true)
y_pred_probs = np.array(y_pred_probs)
y_pred = (y_pred_probs > 0.5).astype(int)

print(classification_report(y_true, y_pred, target_names=['Normal', 'Abnormal']))

cm = confusion_matrix(y_true, y_pred)
print('Confusion Matrix:\n', cm)

fpr, tpr, _ = roc_curve(y_true, y_pred_probs)
roc_auc = auc(fpr, tpr)
plt.plot(fpr, tpr, label=f'AUC = {roc_auc:.3f}')
plt.plot([0,1],[0,1],'k--')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC Curve'); plt.legend()
plt.show()

## 10. Save the model

In [ ]:
model.save('/content/drive/MyDrive/kidney_model.h5')
print('Model saved to Google Drive as kidney_model.h5')
print('Download this file and place it in your Flask app under app/model/kidney_model.h5')